# Convert TOA .SAFE files to .tif

In [ ]:
from pathlib import Path
import re, glob, warnings, numpy as np, geopandas as gpd, rasterio
from rasterio.io import MemoryFile
from rasterio.enums import Resampling
from rasterio.warp import reproject
from rasterio.mask import mask
from rasterio.transform import from_gcps
from rasterio.errors import NotGeoreferencedWarning


In [ ]:
def toa_SAFE_to_TIF(input_dir, clip_shp=None, cloudmasking=False):
    d=Path(input_dir); bands="B01 B02 B03 B04 B05 B06 B07 B08 B8A B09 B10 B11 B12".split()
    safes=sorted([p for p in d.iterdir() if p.is_dir() and p.name.endswith(".SAFE")])
    if not safes: raise FileNotFoundError(f"No .SAFE folders in {d}")
    gdf=gpd.read_file(clip_shp) if clip_shp else None
    if gdf is not None and gdf.crs is None: raise ValueError("clip_shp has no CRS")

    def _granule_from_band(bp):
        p=Path(bp)
        for par in p.parents:
            if par.parent.name=="GRANULE": return par
        raise ValueError(f"Could not find GRANULE folder for {bp}")

    def _res_m(p):
        m=re.search(r"_(\d+)m$", Path(p).stem)
        return int(m.group(1)) if m else 999

    def _band_hits(gran, b):
        return sorted(
            [p for p in (gran/"IMG_DATA").rglob("*.jp2")
             if re.search(fr"_{re.escape(b)}(?:_|$)", p.stem)],
            key=lambda p: (_res_m(p), str(p))
        )

    def _band_path(gran, b, required=True):
        hits=_band_hits(gran, b)
        if not hits:
            if required: raise FileNotFoundError(f"{gran.name} band {b}: no match found")
            return None
        return str(hits[0])   # finest available resolution

    def _one(paths, label):
        paths=sorted(map(Path, paths))
        if not paths: raise FileNotFoundError(f"{label}: no match found")
        if len(paths)>1: print(f"  - {label}: multiple matches, using {paths[0].name}")
        return str(paths[0])

    def _scl_path(gran):
        return _one((gran/"IMG_DATA"/"R20m").glob("*_SCL_20m.jp2"), f"{gran.name} SCL")

    def _qi_path(gran):
        return _one((gran/"QI_DATA").glob("*MSK_CLASSI_B00.jp2"), f"{gran.name} QI mask")

    def warp(p, ref, dt):
        with rasterio.open(p) as src:
            if (src.crs,src.transform,src.width,src.height)==(ref.crs,ref.transform,ref.width,ref.height):
                return src.read(1).astype(dt, copy=False)
            dst=np.empty((ref.height,ref.width), dt)
            reproject(src.read(1), dst, src_transform=src.transform, src_crs=src.crs,
                      dst_transform=ref.transform, dst_crs=ref.crs,
                      dst_width=ref.width, dst_height=ref.height,
                      resampling=Resampling.nearest)
            return dst

    outs=[]
    for safe in safes:
        out=d/(safe.name[:-5]+".TIF")
        print(f"\n=== {safe.name} ===")

        b02_hits=glob.glob(str(safe/"**"/"*B02*.jp2"), recursive=True)
        if not b02_hits: raise FileNotFoundError(f"B02 not found in {safe}")
        gran=_granule_from_band(sorted(b02_hits)[0])
        print(f"• using granule: {gran.name}")

        P={}; use_bands=[]
        for b in bands:
            p=_band_path(gran, b, required=(b!="B10"))
            if p is None:
                print(f"  - {b}: not found, skipping")
                continue
            P[b]=p; use_bands.append(b)
            print(f"  - {b}: {Path(p).parent.name}/{Path(p).name}")

        if "B02" not in P: raise FileNotFoundError(f"B02 not found in {gran}")
        scp=_scl_path(gran)
        mp=_qi_path(gran) if cloudmasking else None
        print("  - SCL:", Path(scp).parent.name+"/"+Path(scp).name)
        if mp: print("  - QI :", Path(mp).parent.name+"/"+Path(mp).name)

        with rasterio.open(P["B02"]) as ref:
            print("• stack to B02 grid (nearest)")
            meta=ref.profile.copy()
            meta.update(driver="GTiff", count=len(use_bands), dtype="int16", nodata=-1)
            mem=MemoryFile(); ds=mem.open(**meta)
            try:
                for i,b in enumerate(use_bands,1):
                    ds.write(warp(P[b], ref, np.int16), i)

                if gdf is not None:
                    print("• clip to AOI")
                    g=gdf if gdf.crs==ds.crs else gdf.to_crs(ds.crs)
                    shapes=[x for x in g.geometry if x is not None and not x.is_empty]
                    data,tr=mask(ds, shapes, crop=True, filled=True)
                    wmeta=ds.meta.copy()
                    wmeta.update(transform=tr, height=data.shape[1], width=data.shape[2])
                else:
                    data=ds.read(); wmeta=ds.meta.copy(); shapes=None

                bad=None

                print("• SCL mask")
                scl_full=warp(scp, ref, np.uint8)
                if shapes is not None:
                    scl_meta=ref.profile.copy()
                    scl_meta.update(driver="GTiff", count=1, dtype="uint8", nodata=0)
                    with MemoryFile() as m:
                        with m.open(**scl_meta) as tmp:
                            tmp.write(scl_full, 1)
                            scl_clip,_=mask(tmp, shapes, crop=True, filled=True)
                    scl=scl_clip[0]
                else:
                    scl=scl_full
                scl_bad=np.isin(scl, [3,8,9,10])
                bad=scl_bad if bad is None else (bad | scl_bad)
                print(f"  - SCL bad px: {int(scl_bad.sum())}")

                if cloudmasking:
                    print("• cloudmask")
                    cm_full=warp(mp, ref, np.uint8)
                    if shapes is not None:
                        cm_meta=ref.profile.copy()
                        cm_meta.update(driver="GTiff", count=1, dtype="uint8", nodata=0)
                        with MemoryFile() as m:
                            with m.open(**cm_meta) as tmp:
                                tmp.write(cm_full, 1)
                                cm_clip,_=mask(tmp, shapes, crop=True, filled=True)
                        cm=cm_clip[0]
                    else:
                        cm=cm_full
                    cm_bad=(cm==1)
                    bad=cm_bad if bad is None else (bad | cm_bad)
                    print(f"  - cloudmask px: {int(cm_bad.sum())}")

                if bad is not None and bad.any():
                    print(f"• apply combined mask across all bands: {int(bad.sum())} px")
                    data[:, bad] = -1

                print("• output band order:", use_bands)
                print(f"• write {out.name}")
                wmeta.update(compress="zstd", predictor=2, tiled=True,
                             blockxsize=512, blockysize=512, interleave="pixel")
                with rasterio.open(out, "w", **wmeta) as dst:
                    dst.write(data)

                outs.append(str(out))
                print("✓ done")
            finally:
                try: ds.close()
                except: pass
                try: mem.close()
                except: pass
    return outs

In [ ]:
CORSICA_processing_dir = r"M:\SDB_Compare\Corsica"
KEYWEST_processing_dir = r"M:\SDB_Compare\Key West"
UNGAVA_processing_dir = r"M:\SDB_Compare\Ungava Bay"
CAPECOD_processing_dir= r"M:\SDB_Compare\Cape Cod"

In [ ]:
CORSICA_shapefile = r"M:\AOI_new\Corsica_AOI.shp"
KEYWEST_shapefile = r"M:\AOI_new\Key_West_AOI_v2.shp"
UNGAVA_shapefile = r"M:\AOI_new\Ungava_Bay_AOI.shp"
CAPECOD_shapefile = r"M:\AOI_new\Cape_Cod_AOI_v2.shp"

# Append SCL data from L2A to L1C tifs and sort

Given a folder of .SAFE images of L1C data, use the date to get the L2 metadata (specifically the SCL band).
Save the .SAFE file as a tif where the SCL band is appended. 

In [ ]:
from pathlib import Path
import os, glob, numpy as np, rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling

In [ ]:


def append_scl_to_tifs(out_dir, overwrite=False, nodata_out=-1):
    d=Path(out_dir)
    tifs=sorted([p for p in d.glob("*.tif") if (d/(p.stem+".SAFE")).is_dir()])
    if not tifs: raise FileNotFoundError(f"No TIFFs with matching SAFE folders found in {d}")

    def _one(paths, label):
        paths=sorted(map(Path, paths))
        if not paths: raise FileNotFoundError(f"{label}: no match found")
        return paths[0]

    def _granule_from_safe(safe):
        b02=_one(glob.glob(str(safe/"**"/"*B02*.jp2"), recursive=True), f"{safe.name} B02")
        for par in b02.parents:
            if par.parent.name=="GRANULE": return par
        raise ValueError(f"Could not find GRANULE folder for {safe.name}")

    def _scl_path(gran):
        return _one((gran/"IMG_DATA").rglob("*_SCL_20m.jp2"), f"{gran.name} SCL")

    def _warp_to_ref(p, ref, dt=np.int16, nodata=nodata_out):
        with rasterio.open(p) as src:
            dst=np.full((ref.height, ref.width), nodata, dtype=dt)
            reproject(
                source=src.read(1),
                destination=dst,
                src_transform=src.transform, src_crs=src.crs,
                dst_transform=ref.transform, dst_crs=ref.crs,
                src_nodata=src.nodata, dst_nodata=nodata,
                resampling=Resampling.nearest
            )
            return dst

    outs=[]
    for tif in tifs:
        safe=d/(tif.stem+".SAFE")
        gran=_granule_from_safe(safe)
        scl_p=_scl_path(gran)

        out=tif if overwrite else tif.with_name(f"{tif.stem}_plusSCL.tif")
        tmp=tif.with_name(f"{tif.stem}__tmp.tif") if overwrite else out

        print(f"\n=== {tif.name} ===")
        print(f"• SAFE: {safe.name}")
        print(f"• granule: {gran.name}")
        print(f"• SCL: {scl_p.name}")

        with rasterio.open(tif) as src:
            nd=src.nodata if src.nodata is not None else nodata_out
            meta=src.meta.copy()
            meta.update(
                driver="GTiff",
                count=src.count+1,
                dtype="int16",
                nodata=nd,
                compress="zstd",
                predictor=2,
                tiled=True,
                blockxsize=512,
                blockysize=512,
                interleave="pixel"
            )

            scl=_warp_to_ref(scl_p, src, np.int16, nd)

            with rasterio.open(tmp, "w", **meta) as dst:
                for b in range(1, src.count+1):
                    dst.write(src.read(b), b)
                    if src.descriptions and src.descriptions[b-1]:
                        dst.set_band_description(b, src.descriptions[b-1])

                dst.write(scl, src.count+1)
                dst.set_band_description(src.count+1, "SCL")

        if overwrite:
            os.replace(tmp, tif)
            outs.append(str(tif))
            print(f"✓ updated in place: {tif.name}")
        else:
            outs.append(str(out))
            print(f"✓ wrote: {out.name}")

    return outs

In [ ]:
CORSICA_processing_dir = r"M:\SDB_Compare\Corsica"
KEYWEST_processing_dir = r"M:\SDB_Compare\Key West"
UNGAVA_processing_dir = r"M:\SDB_Compare\Ungava Bay"
CAPECOD_processing_dir= r"M:\SDB_Compare\Cape Cod"

In [ ]:
processing_dir= CORSICA_processing_dir
outs=append_scl_to_tifs(processing_dir, overwrite=False)
print(outs)

# Mask out the SCL values 3, 8, 9, 10

In [ ]:
import tempfile
import shutil
from pathlib import Path
import numpy as np, rasterio

In [ ]:
def mask_scl_values(
    folder: str,
    scl_values_to_mask: list[int] = [3, 8, 9, 10],
    nodata_value: int = 0,
) -> list[str]:
    """
    Iterate over all .tif files in a folder and mask out specified SCL values.
    Replaces each original file with a masked version, appending '_masked' to the filename.

    Sentinel-2 SCL classes masked by default:
        3  = Cloud shadow
        8  = Cloud medium probability
        9  = Cloud high probability
        10 = Thin cirrus

    Args:
        folder:              Path to folder containing input .tif files.
        scl_values_to_mask:  List of SCL pixel values to mask out.
        nodata_value:        Integer value assigned to masked pixels (default: 0).

    Returns:
        List of output file paths that were successfully written.
    """
    folder = Path(folder)
    tif_files = sorted([
        p for p in folder.glob("*.tif")
        if "masked" not in p.stem.lower()
    ])

    if not tif_files:
        raise FileNotFoundError(f"No .tif files found in: {folder}")

    output_paths = []

    for tif_path in tif_files:
        print(f"Processing: {tif_path.name}")

        # Write to a temp file in the same folder so the original is
        # never touched until we're sure the masked write succeeded
        tmp_path = tif_path.with_suffix(".tmp.tif")

        try:
            with rasterio.open(tif_path) as src:
                band_names = list(src.descriptions) if src.descriptions else []

                scl_band_index = next(
                    (i + 1 for i, name in enumerate(band_names)
                     if name and "SCL" in name.upper()),
                    None,
                )

                if scl_band_index is None:
                    print(f"  WARNING: No SCL band found in {tif_path.name}, skipping.")
                    continue

                scl = src.read(scl_band_index)
                bad_pixels = np.isin(scl, scl_values_to_mask)

                profile = src.profile.copy()
                profile.update(
                    nodata=nodata_value,
                    compress="zstd",
                )

                with rasterio.open(tmp_path, "w", **profile) as dst:
                    for band_idx in range(1, src.count + 1):
                        band_data = src.read(band_idx)
                        band_data[bad_pixels] = nodata_value
                        dst.write(band_data, band_idx)
                        dst.update_tags(band_idx, description=band_names[band_idx - 1])

            # Only remove original and rename once write has succeeded
            out_path = tif_path.with_name(f"{tif_path.stem}_masked{tif_path.suffix}")
            tif_path.unlink()
            tmp_path.rename(out_path)

        except Exception as e:
            # Clean up temp file if something went wrong
            if tmp_path.exists():
                tmp_path.unlink()
            raise RuntimeError(f"Failed processing {tif_path.name}: {e}") from e

        print(f"  Saved → {out_path}")
        output_paths.append(str(out_path))

    print(f"\nDone. {len(output_paths)}/{len(tif_files)} files processed.")
    return output_paths

In [ ]:
CORSICA_processing_dir = r"M:\SDB_Compare\Corsica\with_SCL"
KEYWEST_processing_dir = r"M:\SDB_Compare\Key West\with_SCL"
UNGAVA_processing_dir = r"M:\SDB_Compare\Ungava Bay\with_SCL"
CAPECOD_processing_dir= r"M:\SDB_Compare\Cape Cod\with_SCL"

In [ ]:
# run 
masked_files = mask_scl_values(folder= CORSICA_processing_dir)

# Basic Median Composite

This function takes the file directory containing tif files and produces a single tif with per-band median composites. The output files are named after the file directory (site name) and "median_composite".

In [ ]:
# new function where we explicitly do not include SCL band

def make_median_composite(out_dir, nodata_out=-1):
    import time
    d = Path(out_dir)
    
    # sorting for tif files ending in 'masked' (to confirm previous filtering) and start with S2A/B name convention
    tif_paths = sorted([p for p in d.glob("*_masked.tif") if p.stem.startswith(("S2A_MSIL2A", "S2B_MSIL2A"))])
    
    
    if not tif_paths:
        raise FileNotFoundError(f"No .tif files found in {d}")

    out = d / f"S2_{d.parent.name.replace(' ', '')}_median_composite.tif"
       
    sitename = d.parent.name.replace(" ", "_")
    foldername = d.name.replace(" ", "_")
    out = d / f"{sitename}_{foldername}_median.tif"
    with rasterio.open(tif_paths[0]) as ref:
        meta = ref.meta.copy()
        count, h, w = ref.count, ref.height, ref.width
        grid = (ref.crs, ref.transform, w, h)
        descriptions = list(ref.descriptions)
        meta.update(
            dtype="int16",
            nodata=nodata_out,
            count=12,
            compress="zstd",
            predictor=2,
            tiled=True,
            blockxsize=512,
            blockysize=512,
            interleave="pixel"
        )
    for p in tif_paths[1:]:
        with rasterio.open(p) as src:
            if (src.crs, src.transform, src.width, src.height) != grid or src.count != count:
                raise ValueError(f"Grid/band mismatch: {p.name}")
    t_start = time.time()
    with rasterio.open(out, "w", **meta) as dst, rasterio.open(tif_paths[0]) as ref:
        dst.descriptions = tuple(descriptions[:12])
        for b in range(1, 13):
            print(f"band {b}/12")
            for _, win in ref.block_windows(1):
                stack = []
                for p in tif_paths:
                    with rasterio.open(p) as src:
                        arr = src.read(b, window=win).astype("float32")
                        nd = src.nodata if src.nodata is not None else nodata_out
                        arr[arr == nd] = np.nan
                        stack.append(arr)
                stk = np.stack(stack, 0)
                valid = np.any(~np.isnan(stk), axis=0)
                med = np.full(stk.shape[1:], nodata_out, dtype="float32")
                med[valid] = np.nanmedian(stk[:, valid], axis=0)
                dst.write(np.rint(med).astype("int16"), b, window=win)
    elapsed = time.time() - t_start
    txt_out = out.with_suffix(".txt")
    with open(txt_out, "w") as f:
        f.write(f"output: {out.name}\n")
        f.write(f"n_files: {len(tif_paths)}\n")
        f.write(f"elapsed: {elapsed:.1f}s ({elapsed/60:.2f} min)\n")
    print(f"wrote: {out}")
    print(f"elapsed: {elapsed:.1f}s ({elapsed/60:.2f} min)")
    return str(out)

In [ ]:
CORSICA_processing_dir = r"M:\SDB_Compare\Corsica\with_SCL"
KEYWEST_processing_dir = r"M:\SDB_Compare\Key West\with_SCL"
UNGAVA_processing_dir = r"M:\SDB_Compare\Ungava Bay\with_SCL"
CAPECOD_processing_dir= r"M:\SDB_Compare\Cape Cod\with_SCL"

In [ ]:
# run
processing_dir=CORSICA_processing_dir
comp=make_median_composite(processing_dir)
print(comp)

# Maximum Outlier Removal Mean Composite

This function takes the file directory and a user defined standard deviation threshold. 

On a per-band basis, the mean and standard deviation of the band is defined from the collection, and values falling outside this are removed from consideration. The remaining values are used to create a median composite. The output is a single tif file with a per-band maximum-outlier-removed mean, named after the file directory of the site, and the standard deviation threshold. 

In [ ]:
# new fuction with SCL exclusion

def make_3sd_filtered_mean_composite(out_dir, n_std, nodata_out=-1):
    import time
    d = Path(out_dir)
    tif_paths = sorted([p for p in d.glob("*_masked.tif") if p.stem.startswith(("S2A_MSIL2A", "S2B_MSIL2A"))])
    if not tif_paths:
        raise FileNotFoundError(f"No .tif files found in {d}")
    tag = str(n_std).replace(".", "p")
    out = d / f"S2_{d.parent.name.replace(' ', '')}_max_outlier_{tag}sd_mean.tif"
    with rasterio.open(tif_paths[0]) as ref:
        meta = ref.meta.copy()
        count, h, w = ref.count, ref.height, ref.width
        grid = (ref.crs, ref.transform, w, h)
        descriptions = list(ref.descriptions)
        meta.update(
            dtype="int16",
            nodata=nodata_out,
            count=12,
            compress="zstd",
            predictor=2,
            tiled=True,
            blockxsize=512,
            blockysize=512,
            interleave="pixel"
        )
    for p in tif_paths[1:]:
        with rasterio.open(p) as src:
            if (src.crs, src.transform, src.width, src.height) != grid or src.count != count:
                raise ValueError(f"Grid/band mismatch: {p.name}")
    t_start = time.time()
    with rasterio.open(out, "w", **meta) as dst, rasterio.open(tif_paths[0]) as ref:
        dst.descriptions = tuple(descriptions[:12])
        for b in range(1, 13):
            print(f"band {b}/12")
            for _, win in ref.block_windows(1):
                stack = []
                for p in tif_paths:
                    with rasterio.open(p) as src:
                        arr = src.read(b, window=win).astype("float32")
                        nd = src.nodata if src.nodata is not None else nodata_out
                        arr[arr == nd] = np.nan
                        stack.append(arr)
                stk = np.stack(stack, 0)
                mu = np.nanmean(stk, axis=0)
                sd = np.nanstd(stk, axis=0)
                keep = (~np.isnan(stk)) & (np.abs(stk - mu) <= (n_std * sd))
                keep |= (~np.isnan(stk)) & (sd == 0) & (stk == mu)
                filt = np.where(keep, stk, np.nan)
                valid = np.any(~np.isnan(filt), axis=0)
                result = np.full(filt.shape[1:], nodata_out, dtype="float32")
                result[valid] = np.nanmean(filt[:, valid], axis=0)
                dst.write(np.rint(result).astype("int16"), b, window=win)
    elapsed = time.time() - t_start
    txt_out = out.with_suffix(".txt")
    with open(txt_out, "w") as f:
        f.write(f"output: {out.name}\n")
        f.write(f"n_files: {len(tif_paths)}\n")
        f.write(f"elapsed: {elapsed:.1f}s ({elapsed/60:.2f} min)\n")
    print(f"wrote: {out}")
    print(f"elapsed: {elapsed:.1f}s ({elapsed/60:.2f} min)")
    return str(out)

In [ ]:
CORSICA_processing_dir = r"M:\SDB_Compare\Corsica\with_SCL"
KEYWEST_processing_dir = r"M:\SDB_Compare\Key West\with_SCL"
UNGAVA_processing_dir = r"M:\SDB_Compare\Ungava Bay\with_SCL"
CAPECOD_processing_dir= r"M:\SDB_Compare\Cape Cod\with_SCL"

In [ ]:
# run
processing_dir=CORSICA_processing_dir
comp=make_3sd_filtered_mean_composite(processing_dir, n_std=3)
print(comp)

# Coastal Clean Water Composite

This function will take the the file directory for tthe site (containing the per-made tif files) and apply additional filtering based on SCL values 4 & 5, and thresholds for B9 (>50 and <300) and B3 (>100) to produce a coastal clean water composite congruent with the Java Script code from Li et al. (2021). The standard marking performed on creation of the tifs filtered for SCL values not equal to 3, 8, 9, and 10.

In [ ]:
from scipy.ndimage import binary_erosion

def mask_s2_noqa_folder(in_dir, out_subdir="clean_coastal_masked", nodata_out=-1.0, band_map=None, overwrite=False):
    d = Path(in_dir)
    out_dir = d / out_subdir
    out_dir.mkdir(exist_ok=True)
    tif_paths = sorted([p for p in d.glob("*_masked.tif") if p.is_file()])
    if not tif_paths:
        raise FileNotFoundError(f"No _masked.tif files found in {d}")
    if band_map is None:
        band_map = {
            "B01":1, "B02":2, "B03":3, "B04":4, "B05":5, "B06":6, "B07":7,
            "B08":8, "B8A":9, "B09":10, "B11":11, "B12":12, "SCL":13
        }
    foot = np.array([[0,1,0],
                     [1,1,1],
                     [0,1,0]], dtype=bool)
    outs = []
    for tif in tif_paths:
        out = out_dir / f"{tif.stem}_cleancoastal.tif"
        if out.exists() and not overwrite:
            print(f"skip exists: {out.name}")
            outs.append(str(out))
            continue
        with rasterio.open(tif) as src:
            arr = src.read().astype("float32")
            nd = src.nodata if src.nodata is not None else nodata_out
            def band(name):
                return arr[band_map[name]-1]
            b3  = band("B03")
            b5  = band("B05")
            b8  = band("B08")
            b9  = band("B09")
            scl = band("SCL")
            ma = (b3!=nd) & (b5!=nd) & (b8!=nd) & (b9!=nd) & (scl!=nd)
            # SCL exclusions
            # ma &= (scl!=3)  # already filtered in mask_scl_values
            ma &= (scl!=4)
            ma &= (scl!=5)
            # ma &= (scl!=8)  # already filtered in mask_scl_values
            # ma &= (scl!=9)  # already filtered in mask_scl_values
            # ma &= (scl!=10) # already filtered in mask_scl_values
            # spectral thresholds (+1000 added)
            ma &= (b9 < 1300)
            ma &= (b9 > 1050)
            ma &= (b3 > 1100)
            # focal_min(circle(radius=1), iterations=1) approximation
            ma = binary_erosion(ma, structure=foot, iterations=1, border_value=0)
            # bad data filters (+1000 added)
            ma &= (b5 < 2000)
            ma &= (b8 < 1300)
            # ndwi_revise = (B3 - B8) / (B3 + B8)
            denom = b3 + b8
            ndwi = np.full(b3.shape, np.nan, dtype="float32")
            ok = denom != 0
            ndwi[ok] = (b3[ok] - b8[ok]) / denom[ok]
            ma &= (ndwi > 0)
            arr[:, ~ma] = nodata_out
            meta = src.meta.copy()
            meta.update(
                dtype="float32",
                nodata=nodata_out,
                compress="zstd",
                predictor=2,
                tiled=True,
                blockxsize=512,
                blockysize=512,
                interleave="pixel"
            )
            with rasterio.open(out, "w", **meta) as dst:
                dst.write(arr)
                if src.descriptions:
                    for i, desc in enumerate(src.descriptions, 1):
                        if desc:
                            dst.set_band_description(i, desc)
        outs.append(str(out))
        print(f"wrote: {out.name}")
    return outs

then a median reducer 

In [ ]:
def make_cc_median_composite(out_dir, nodata_out=-1):
    import time
    d = Path(out_dir)
    tif_paths = sorted([p for p in d.glob("*.tif") if "median_composite" not in p.stem.lower()])
    if not tif_paths:
        raise FileNotFoundError(f"No .tif files found in {d}")
    out = d / f"S2_{d.parent.parent.name.replace(' ', '')}_cleancoastal_median_composite.tif"
    with rasterio.open(tif_paths[0]) as ref:
        meta = ref.meta.copy()
        count, h, w = ref.count, ref.height, ref.width
        grid = (ref.crs, ref.transform, w, h)
        descriptions = list(ref.descriptions)
        meta.update(
            dtype="int16",
            nodata=nodata_out,
            count=12,
            compress="zstd",
            predictor=2,
            tiled=True,
            blockxsize=512,
            blockysize=512,
            interleave="pixel"
        )
    for p in tif_paths[1:]:
        with rasterio.open(p) as src:
            if (src.crs, src.transform, src.width, src.height) != grid or src.count != count:
                raise ValueError(f"Grid/band mismatch: {p.name}")
    t_start = time.time()
    with rasterio.open(out, "w", **meta) as dst, rasterio.open(tif_paths[0]) as ref:
        dst.descriptions = tuple(descriptions[:12])
        for b in range(1, 13):
            print(f"band {b}/12")
            for _, win in ref.block_windows(1):
                stack = []
                for p in tif_paths:
                    with rasterio.open(p) as src:
                        arr = src.read(b, window=win).astype("float32")
                        nd = src.nodata if src.nodata is not None else nodata_out
                        arr[arr == nd] = np.nan
                        stack.append(arr)
                stk = np.stack(stack, 0)
                valid = np.any(~np.isnan(stk), axis=0)
                med = np.full(stk.shape[1:], nodata_out, dtype="float32")
                med[valid] = np.nanmedian(stk[:, valid], axis=0)
                dst.write(np.rint(med).astype("int16"), b, window=win)
    elapsed = time.time() - t_start
    txt_out = out.with_suffix(".txt")
    with open(txt_out, "w") as f:
        f.write(f"output: {out.name}\n")
        f.write(f"n_files: {len(tif_paths)}\n")
        f.write(f"elapsed: {elapsed:.1f}s ({elapsed/60:.2f} min)\n")
    print(f"wrote: {out}")
    print(f"elapsed: {elapsed:.1f}s ({elapsed/60:.2f} min)")
    return str(out)

In [ ]:
CORSICA_processing_dir = r"M:\SDB_Compare\Corsica\with_SCL"
KEYWEST_processing_dir = r"M:\SDB_Compare\Key West\with_SCL"
UNGAVA_processing_dir = r"M:\SDB_Compare\Ungava Bay\with_SCL"
CAPECOD_processing_dir= r"M:\SDB_Compare\Cape Cod\with_SCL"

In [ ]:
# run clean coastal filtering 
processing_dir=KEYWEST_processing_dir
outs=mask_s2_noqa_folder(processing_dir, overwrite=True)
print(outs)

Then get the median reducer on the "Clean Coastal" filtered data:

In [ ]:
# run median composite
processing_dir=r"M:\SDB_Compare\Corsica\with_SCL\clean_coastal_masked"
comp=make_cc_median_composite(processing_dir)
print(comp)

# 25th Percentile Composite

For each band, this function will consider the distribution of the values of a given pixel through the designated collection, and retain only those with the lowest (25th percentile) reflectance values. The median of these 25th percentile values will be used to create the resulting composite. 

In [ ]:
def make_25p_composite(out_dir, nodata_out=-1):
    import time
    d = Path(out_dir)
    tif_paths = sorted([p for p in d.glob("S2*_masked.tif") if "composite" not in p.stem.lower()])
    if not tif_paths:
        raise FileNotFoundError(f"No S2 .tif files found in {d}")
    out = d / f"S2_{d.parent.name.replace(' ', '')}_p25_composite.tif"
    with rasterio.open(tif_paths[0]) as ref:
        meta = ref.meta.copy()
        count, h, w = ref.count, ref.height, ref.width
        grid = (ref.crs, ref.transform, w, h)
        meta.update(
            dtype="int16",
            nodata=nodata_out,
            compress="zstd",
            predictor=2,
            tiled=True,
            blockxsize=512,
            blockysize=512,
            interleave="pixel"
        )
    for p in tif_paths[1:]:
        with rasterio.open(p) as src:
            if (src.crs, src.transform, src.width, src.height) != grid or src.count != count:
                raise ValueError(f"Grid/band mismatch: {p.name}")
    t_start = time.time()
    with rasterio.open(out, "w", **meta) as dst, rasterio.open(tif_paths[0]) as ref:
        for b in range(1, count + 1):
            print(f"band {b}/{count}")
            for _, win in ref.block_windows(b):
                stack = []
                for p in tif_paths:
                    with rasterio.open(p) as src:
                        arr = src.read(b, window=win).astype("float32")
                        nd = src.nodata if src.nodata is not None else nodata_out
                        arr[arr == nd] = np.nan
                        stack.append(arr)
                stk = np.stack(stack, 0)
                any_valid = np.any(~np.isnan(stk), axis=0)
                med = np.full(stk.shape[1:], nodata_out, dtype="float32")
                if any_valid.any():
                    stk_v = stk[:, any_valid]
                    p25 = np.nanpercentile(stk_v, 25, axis=0)
                    keep = (~np.isnan(stk_v)) & (stk_v <= p25)
                    filt = np.where(keep, stk_v, np.nan)
                    has_filt = np.any(~np.isnan(filt), axis=0)
                    med[any_valid] = np.where(
                        has_filt,
                        np.nanmedian(filt, axis=0),
                        nodata_out
                    )
                dst.write(np.rint(med).astype("int16"), b, window=win)
    elapsed = time.time() - t_start
    txt_out = out.with_suffix(".txt")
    with open(txt_out, "w") as f:
        f.write(f"output: {out.name}\n")
        f.write(f"n_files: {len(tif_paths)}\n")
        f.write(f"elapsed: {elapsed:.1f}s ({elapsed/60:.2f} min)\n")
    print(f"wrote: {out}")
    print(f"elapsed: {elapsed:.1f}s ({elapsed/60:.2f} min)")
    return str(out)

In [ ]:
CORSICA_processing_dir = r"M:\SDB_Compare\Corsica\with_SCL"
KEYWEST_processing_dir = r"M:\SDB_Compare\Key West\with_SCL"
UNGAVA_processing_dir = r"M:\SDB_Compare\Ungava Bay\with_SCL"
CAPECOD_processing_dir= r"M:\SDB_Compare\Cape Cod\with_SCL"

In [ ]:
# run 25th percentile median composite
out_dir= CORSICA_processing_dir
output_file = make_25p_composite(out_dir)
print("Composite saved to:", output_file)